# Visual Product Search Engine
## DeepFashion In-Shop Clothes Retrieval

**Architecture**: YOLO → BLIP-2 → CLIP (fine-tuned) → HNSW FAISS  
**Conditions**: A (vision-only) | B (frozen CLIP + BLIP-2) | C (fine-tuned CLIP + BLIP-2)


In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
import subprocess, sys

packages = [
    'open-clip-torch',
    'faiss-gpu',
    'transformers>=4.37',
    'ultralytics',
    'accelerate',
    'streamlit',
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('All packages installed.')

In [ ]:
# ── 2. Clone / mount project code ────────────────────────────────────────────
import os, sys

# If running from GitHub clone:
# !git clone https://github.com/<your-repo>/visual-search.git /kaggle/working/visual_search

# Otherwise upload the src/ scripts/ app/ folders to /kaggle/working/ manually.
PROJECT_ROOT = '/kaggle/working/visual_search'
sys.path.insert(0, PROJECT_ROOT)

DATASET_ROOT = '/kaggle/input/deepfashion-inshop'   # adjust to your dataset path
OUTPUT_DIR   = '/kaggle/working'
CKPT_DIR     = f'{OUTPUT_DIR}/checkpoints'
INDEX_DIR    = f'{OUTPUT_DIR}/index'
RESULTS_DIR  = f'{OUTPUT_DIR}/results'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Dataset root: {DATASET_ROOT}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# ── 3. Verify dataset files ───────────────────────────────────────────────────
import os

required = [
    'list_eval_partition.txt',
    'list_description_inshop.txt',
    'list_bbox_inshop.txt',
    'img',
]
for f in required:
    path = os.path.join(DATASET_ROOT, f)
    exists = os.path.exists(path)
    print(f'  [{"✓" if exists else "✗"}] {f}')

# Quick count
from src.dataset import parse_eval_partition, parse_item_ids, parse_bboxes
splits = parse_eval_partition(f'{DATASET_ROOT}/list_eval_partition.txt')
for k, v in splits.items():
    print(f'  {k:8s}: {len(v):,} images')

In [ ]:
# ── 4. STEP A: Build vision-only index (Condition A baseline) ─────────────────
!python {PROJECT_ROOT}/scripts/build_index.py \
    --dataset_root {DATASET_ROOT} \
    --index_dir    {INDEX_DIR} \
    --condition    A \
    --alpha        1.0 \
    --batch_size   64 \
    --embed_dim    256

In [ ]:
# ── 5. Fine-tune CLIP (Condition C) ──────────────────────────────────────────
# Seeds for ablation: use roll numbers or the values below
SEEDS = [42, 2024, 1337, 7]

for seed in SEEDS[:1]:  # Train one seed first; replicate for others
    print(f'\n===== Training seed {seed} =====')
    !python {PROJECT_ROOT}/scripts/train_clip.py \
        --dataset_root    {DATASET_ROOT} \
        --output_dir      {CKPT_DIR}/seed_{seed} \
        --epochs          10 \
        --batch_size      128 \
        --lr              1e-4 \
        --unfreeze_last_n 4 \
        --embed_dim       256 \
        --seed            {seed} \
        --eval_every      2

In [ ]:
# ── 6. Build indices for conditions B and C ───────────────────────────────────
CKPT_PATH = f'{CKPT_DIR}/seed_42/clip_finetuned_best.pt'
ALPHA_B   = 0.6
ALPHA_C   = 0.6

# Condition B: frozen CLIP + BLIP-2 captions
!python {PROJECT_ROOT}/scripts/build_index.py \
    --dataset_root {DATASET_ROOT} \
    --index_dir    {INDEX_DIR} \
    --condition    B \
    --alpha        {ALPHA_B} \
    --batch_size   64 \
    --embed_dim    256

# Condition C: fine-tuned CLIP + BLIP-2 captions
!python {PROJECT_ROOT}/scripts/build_index.py \
    --dataset_root {DATASET_ROOT} \
    --ckpt_path    {CKPT_PATH} \
    --index_dir    {INDEX_DIR} \
    --condition    C \
    --alpha        {ALPHA_C} \
    --batch_size   64 \
    --embed_dim    256

In [ ]:
# ── 7. Full ablation evaluation ───────────────────────────────────────────────
!python {PROJECT_ROOT}/scripts/evaluate.py \
    --dataset_root {DATASET_ROOT} \
    --index_base   {INDEX_DIR} \
    --ckpt_path    {CKPT_PATH} \
    --output_dir   {RESULTS_DIR} \
    --seeds        42 2024 1337 7 \
    --batch_size   64 \
    --alpha_B      {ALPHA_B} \
    --alpha_C      {ALPHA_C} \
    --use_itm

In [ ]:
# ── 8. Display results table ──────────────────────────────────────────────────
import json, pandas as pd

with open(f'{RESULTS_DIR}/ablation_results.json') as f:
    res = json.load(f)

rows = []
for cond, metrics in res.items():
    row = {'Condition': cond.replace('condition_', '')}
    for metric, vals in metrics.items():
        row[metric] = f"{vals['mean']:.4f} ± {vals['std']:.4f}"
    rows.append(row)

df = pd.DataFrame(rows).set_index('Condition')
print(df.to_string())
df

In [ ]:
# ── 9. Launch Streamlit demo (background) ─────────────────────────────────────
# On Kaggle you can expose the Streamlit port via ngrok or run locally.
# Uncomment the block below:

# !pip install -q pyngrok
# from pyngrok import ngrok
# import subprocess, threading, time

# def run_streamlit():
#     subprocess.run([
#         'streamlit', 'run',
#         f'{PROJECT_ROOT}/app/demo.py',
#         '--server.port', '8501',
#         '--server.headless', 'true',
#         '--',
#         '--dataset_root', DATASET_ROOT,
#         '--index_dir',    f'{INDEX_DIR}/condition_C_alpha0.6',
#         '--ckpt_path',    CKPT_PATH,
#     ])

# t = threading.Thread(target=run_streamlit, daemon=True)
# t.start()
# time.sleep(5)
# public_url = ngrok.connect(8501)
# print(f'Demo URL: {public_url}')
print('Uncomment the block above to launch the Streamlit demo.')